# Applying ePMV setpoints

## Introduction

This notebook shows how to use `accim` to apply setpoints based on the **PMV with the expectancy factor (ePMV)** of Fanger and Toftum (2002) to an EnergyPlus model.

In buildings without air-conditioning in warm climates, the occupants feel the warmth as less severe than the PMV of Fanger predicts: they expect warmer indoor conditions, and they slow down in the warmth, so the metabolic rate used to calculate the PMV is overestimated. Fanger and Toftum extended the PMV with an expectancy factor $e$, between 0.5 and 1:

$$ePMV = e \cdot PMV$$

and they reduced the metabolic rate by 6.7 % for each unit of PMV above neutral. The expectancy factor only affects the upper limit of the comfort zone (the warm sensations), so `accim` applies it in the cooling season and to the cooling setpoint (section 4.8.1 of the detailed use).

## Objectives of this Notebook

1. **Setup:** Load a residential EnergyPlus model and prepare it for simulation.
2. **Expectancy factor:** Choose $e$ for the climate of the weather file.
3. **Base case (PMV):** Simulate the building with PMV setpoints (`apply_pmv_setpoints`).
4. **ePMV case:** Simulate the building with ePMV setpoints (`apply_epmv_setpoints`).
5. **ePMV case with the metabolic correction:** Simulate the building with the reduction of the activity level of the occupants (`metabolic_correction=True`).
6. **Compare:** Analyze the setpoints of the thermostats, the PMV of the occupants and the HVAC electricity of the three cases.

## 1. Setup and Imports

We import the necessary libraries for simulation (`besos`, `accim`) and data analysis (`pandas`, `matplotlib`, `seaborn`).

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from besos import eppy_funcs as ef
from besos import eplus_funcs as ep

# The accim module of the PMV-based setpoints, and the constants of the ePMV
import accim.sim.apmv as apmv
from accim.constants import EPMV_EXPECTANCY_RANGES, EPMV_MET_REDUCTION_PER_PMV

# Visualization setup
plt.rcParams['figure.figsize'] = [14, 6]
sns.set_style("whitegrid")

# Define file paths
idf_filename = "TestModel_TestResidentialUnit.idf"
epw_filename = 'Seville.epw'

for filename in (idf_filename, epw_filename):
    print(f"{filename}: {'found' if os.path.exists(filename) else 'NOT FOUND'}")

## 2. Loading the Model

We use BESOS to load the IDF file (the functions of `accim.sim.apmv` also take the IDF objects of eppy). The model has two floors with a VRF system, and one `People` object for both floors. The function `get_available_target_names` lists the occupied zones, whose names are the keys of the dictionaries of arguments (to give a value to each zone).

In [ ]:
building = ef.get_building(idf_filename)

target_names = apmv.get_available_target_names(building=building)
print(f"Found {len(target_names)} targets:")
for t in target_names:
    print(f" - {t}")

## 3. The Expectancy Factor

Fanger and Toftum (2002, Table 1) give the expectancy factor by the expectation of the occupants:

| Expectation | Where | $e$ |
|---|---|---|
| High | Regions where air-conditioned buildings are common; warm periods occurring briefly during the summer | 0.9 to 1.0 |
| Moderate | Regions with some air-conditioned buildings; warm summer | 0.7 to 0.9 |
| Low | Regions with few air-conditioned buildings; warm weather during all seasons | 0.5 to 0.7 |

In their validation, $e$ was 0.6 in Bangkok, 0.7 in Singapore and Athens, and 0.9 in Brisbane. In air-conditioned buildings, $e = 1$, and the ePMV is the PMV. `accim.constants` has these ranges and the reduction of the metabolic rate:

In [ ]:
for expectation, (low, high) in EPMV_EXPECTANCY_RANGES.items():
    print(f"{expectation:>8} expectation: e between {low} and {high}")
print(f"Reduction of the metabolic rate for each unit of PMV above neutral: {EPMV_MET_REDUCTION_PER_PMV:.1%}")

Seville has hot summers, and some of its buildings are air-conditioned: a moderate expectation, with $e$ between 0.7 and 0.9. We use **$e = 0.7$**, the lower end of that range, in the cooling season, from 1 May to 30 September (`cooling_season_end` is the day after the last day of the season).

`pmv_cooling_setpoint` and `pmv_heating_setpoint` are on the ePMV scale. The thermostat controls the PMV, so in the cooling season it receives the PMV whose ePMV is the cooling setpoint, $PMV = ePMV / e$: with $e = 0.7$, an ePMV setpoint of 0.5 (0.4 with the default tolerance of -0.1) becomes a PMV setpoint of 0.714 (0.571). The heating setpoint is not affected: it is a PMV setpoint. In the heating season, $e = 1$.

The three cases use the same setpoints and the same inputs of the PMV of the occupants (100 W/person, 0.1 m/s and a work efficiency of 0, instead of the 0.3 m/s and 0.3 of the model), and the occupants are always present, so that the setpoints apply all the time. The inputs of the PMV are set before the setpoints, since the metabolic correction reads the activity level of the occupants.

In [ ]:
# The PMV inputs of the occupants, the same in the three cases
pmv_inputs = {'activity_level': 100, 'air_velocity': 0.1, 'work_efficiency': 0.0}
# The setpoints, on the scale of the index of each case
setpoint_arguments = dict(
    pmv_cooling_setpoint=0.5,
    pmv_heating_setpoint=-0.5,
    cooling_season_start='01/05',
    cooling_season_end='01/10',
    output_frequencies=['hourly'],
    verbose=False,
)
expectancy_factor = 0.7


def prepare(idf_building):
    # Occupants always present, and the same inputs of the PMV
    apmv.set_zones_always_occupied(idf_building, verbose=False)
    apmv.set_pmv_input_parameters(building=idf_building, verbose=False, **pmv_inputs)
    return idf_building

## 4. Base Case: PMV Setpoints

`apply_pmv_setpoints` keeps the PMV of each zone between the setpoints, with their tolerances: -0.4 and 0.4. They are the setpoints of the category II of EN 16798-1 and ISO 7730 (-0.5 and 0.5; category B in the edition of 2005 of ISO 7730), narrowed by the tolerances.

In [ ]:
building_pmv = prepare(ef.get_building(idf_filename))
apmv.apply_pmv_setpoints(building=building_pmv, **setpoint_arguments)

out_dir_pmv = 'sim_results_pmv'
print(f"Running the base case (PMV) in {out_dir_pmv}...")
ep.run_building(building=building_pmv, out_dir=out_dir_pmv, epw=epw_filename)
print("PMV simulation finished.")

## 5. ePMV Setpoints

`apply_epmv_setpoints` takes the expectancy factor of all the zones, or a dictionary with a value for each zone (see `get_input_template_dictionary`). Its EMS program `apply_ePMV_<zone>` sets `e_<zone>` (the expectancy factor in the cooling season, and 1 in the heating season) and divides the cooling setpoint by it:

In [ ]:
building_epmv = prepare(ef.get_building(idf_filename))
apmv.apply_epmv_setpoints(building=building_epmv, expectancy_factor=expectancy_factor, **setpoint_arguments)

# The EMS program of the first zone (accim replaces spaces and colons with underscores in the EMS names)
target_sanitized = target_names[0].replace(" ", "_").replace(":", "_")
print(building_epmv.getobject('EnergyManagementSystem:Program', f'apply_ePMV_{target_sanitized}'))

In [ ]:
out_dir_epmv = 'sim_results_epmv'
print(f"Running the ePMV case in {out_dir_epmv}...")
ep.run_building(building=building_epmv, out_dir=out_dir_epmv, epw=epw_filename)
print("ePMV simulation finished.")

## 6. ePMV Setpoints with the Metabolic Correction

With `metabolic_correction=True`, the EMS program `correct_activity_<people>` reduces the activity level of the occupants in the cooling season by 6.7 % for each unit of PMV above neutral (up to 3). EnergyPlus calculates the PMV once per timestep, so the correction uses the PMV of the previous timestep. The activity level also gives the heat gains of the occupants, which the correction reduces too.

The `People` object of this model applies to both floors, so it has one activity level, which is reduced with the mean PMV of its occupants in both floors: `accim` warns about it.

In [ ]:
building_met = prepare(ef.get_building(idf_filename))
apmv.apply_epmv_setpoints(building=building_met, expectancy_factor=expectancy_factor, metabolic_correction=True,
                          **setpoint_arguments)

# The EMS program of the metabolic correction
for program in building_met.idfobjects['EnergyManagementSystem:Program']:
    if program.Name.startswith('correct_activity_'):
        print(program)

In [ ]:
out_dir_met = 'sim_results_epmv_met'
print(f"Running the ePMV case with the metabolic correction in {out_dir_met}...")
ep.run_building(building=building_met, out_dir=out_dir_met, epw=epw_filename)
print("ePMV simulation with the metabolic correction finished.")

## 7. Results

We load the hourly results of the three simulations. The CSV files may also have the results of the design days, before those of the run period, so we keep the last 8760 hours (the year 2006 of the run period).

In [ ]:
cases = {
    'PMV': out_dir_pmv,
    'ePMV': out_dir_epmv,
    'ePMV with metabolic correction': out_dir_met,
}
results = {}
for case, folder in cases.items():
    df = pd.read_csv(os.path.join(folder, 'eplusout.csv'))
    results[case] = df.iloc[-8760:].reset_index(drop=True)


def column(df, *texts):
    # The (first) column whose name has all the texts
    matches = [c for c in df.columns if all(text in c for text in texts)]
    if not matches:
        raise KeyError(f'No column with {texts}')
    return df[matches[0]]


# The outputs of the occupants of the first floor have the name of its target, in capitals
people_key = target_names[0].upper()
week = slice(24 * 195, 24 * 202)            # 15 to 21 July (days 196 to 202 of the year)
cooling_season = slice(24 * 120, 24 * 273)  # 1 May to 30 September

### 7.1 The Setpoints of the Thermostats and the PMV in a Summer Week

The dashed lines are the PMV setpoints of the cooling thermostat of the first floor (`PMV_C_SP_<zone>`, with the tolerance): 0.4 in the base case, and 0.4/0.7 = 0.571 with the ePMV setpoints. The solid lines are the PMV of its occupants (the Fanger model of EnergyPlus).

In [ ]:
colors = {'PMV': 'gray', 'ePMV': 'tab:blue', 'ePMV with metabolic correction': 'tab:orange'}
fig, ax = plt.subplots(figsize=(15, 6))
for case, df in results.items():
    pmv = column(df, people_key, 'Fanger Model PMV').iloc[week]
    setpoint = column(df, 'PMV_C_SP_FLOOR_1_ZONE', 'Schedule Value').iloc[week]
    ax.plot(pmv.index, pmv.values, color=colors[case], label=f'PMV, {case}')
    ax.plot(setpoint.index, setpoint.values, color=colors[case], linestyle='--',
            label=f'Cooling PMV setpoint, {case}')
ax.set_xlabel('Hour of the year')
ax.set_ylabel('PMV')
ax.set_title(f'{target_names[0]}: PMV and cooling PMV setpoints, 15 to 21 July')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.35), ncol=3)
plt.show()

### 7.2 The Activity Level with the Metabolic Correction

The outputs `Metabolic Correction Factor_<people>` and `Corrected Activity Level_<people>` report the reduction of the activity level (100 W/person without it).

In [ ]:
df_met = results['ePMV with metabolic correction']
factor = column(df_met, 'Metabolic Correction Factor')
activity = column(df_met, 'Corrected Activity Level')

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(activity.iloc[week].index, activity.iloc[week].values, color='tab:orange')
ax.set_xlabel('Hour of the year')
ax.set_ylabel('Activity level (W/person)')
ax.set_title('Corrected activity level of the occupants, 15 to 21 July')
plt.show()

print(f"Mean reduction of the activity level in the cooling season: {1 - factor.iloc[cooling_season].mean():.1%}")
print(f"Largest reduction: {1 - factor.min():.1%}")

### 7.3 Comfort and HVAC Electricity

For each case: the annual HVAC electricity (the meter `Electricity:HVAC`, in J, converted to kWh), and, in the cooling season, the mean PMV of the occupants of the first floor and the percentage of hours with a PMV above 0.5 (warmer than the category II of EN 16798-1 and ISO 7730).

In [ ]:
summary = {}
for case, df in results.items():
    pmv = column(df, people_key, 'Fanger Model PMV').iloc[cooling_season]
    summary[case] = {
        'HVAC electricity (kWh)': column(df, 'Electricity:HVAC').sum() / 3.6e6,
        'Mean PMV, cooling season': pmv.mean(),
        'Hours with PMV > 0.5, cooling season (%)': 100 * (pmv > 0.5).mean(),
    }
pd.DataFrame(summary).T.round(2)

## 8. Conclusion

This notebook applied the setpoints of the module `apmv` of `accim` based on the PMV and the ePMV:

1. The ePMV setpoints keep the ePMV of each zone between the setpoints. In the cooling season, the PMV setpoint of the cooling thermostat is the ePMV setpoint divided by the expectancy factor (0.571 instead of 0.4 with $e = 0.7$), so the zones are allowed to be warmer, as the figure of section 7.1 shows; the heating setpoint, and all the setpoints in the heating season, are those of the base case.
2. The metabolic correction reduces the activity level of the occupants when they feel warm, so their PMV is lower at the same temperature, and the thermostat allows a higher temperature; it also reduces their heat gains.
3. The table of section 7.3 gives the effect on the comfort of the occupants (on the PMV scale) and on the HVAC electricity. Lower expectancy factors move the cooling setpoint further.

Section 4.8.1 of the detailed use explains the arguments of `apply_epmv_setpoints`, and the approximation of the metabolic correction.

## 9. Cleanup

Finally, we remove the simulation output directories to clean up the workspace. **Warning:** This will permanently delete the simulation results generated in this notebook.

In [ ]:
import shutil

for folder in [out_dir_pmv, out_dir_epmv, out_dir_met]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Removed directory: {folder}")
    else:
        print(f"Directory {folder} not found.")